## CSCE 676 :: Data Mining and Analysis :: Texas A&M University :: Spring 2026

# Weekly Homework 11: Graph Embeddings and Finding Similar Items

***Goals of this homework:***
- Implement and trace through a core graph embedding methods from lecture: **node2vec**.
- Build an intuition for how biased random walks (p, q parameters) control the BFS/DFS tradeoff in node2vec.
- Implement the full **LSH** to efficiently find similar congress members.
- Reason about the tradeoffs between graph embedding approaches and the correctness/efficiency tradeoffs inherent to LSH.

***Submission instructions:***

Post your notebook to Canvas as **your-uin_hw11.ipynb**. Run all cells before submitting so we can see the output.

***Grading philosophy:***

We are grading reasoning, judgment, and clarity, not just correctness. Show us that you understand the data, the constraints, and the limits of your conclusions.

***For each question, respond with 2 cells:***
1. **[A Code Cell] Your Code** — implement the task, then include **2–3 assertions or print-based checks** directly in this cell that verify your implementation is correct. Choose tests that would catch the most common bugs.
2. **[A Markdown Cell] Your Answer** — write up your answers in complete sentences. Explain what tests you wrote and **why those specific tests prove your code is correct** (not just that it ran).

***At the end of each Section (A/B/C) include a resources cell:***

```
On my honor, I declare the following resources:
1. Collaborators:
- ...
2. Web Sources:
- ...
3. AI Tools:
- ...
```

---
## Preliminaries: Setup

Run the cells below once before starting. They install dependencies, download the Congress Twitter Network, and build a NetworkX graph. You do not need to modify these cells.

**Dataset:** The [Congress Twitter Network](https://snap.stanford.edu/data/congress-twitter.html) from SNAP. Each node is a member of the 117th US Congress. A directed edge A → B means A **follows** B on Twitter. Edge weights represent estimated information transmission probability.

In [1]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install torch torch_geometric -q
!pip install torch_scatter torch_sparse torch_cluster     -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cpu.html -q
!pip install datasketch -q
!pip install gensim

# ── Download dataset ──────────────────────────────────────────────────────────
!curl -sL -o congress_network.zip https://snap.stanford.edu/data/congress_network.zip
!unzip -qo congress_network.zip


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Using cached smart_open-7.5.1-py3-none-any.whl.metadata (24 kB)
  Using cached wrapt-2.1.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (7.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 24.7 MB/s eta 0:00:00m eta 0:00:010:00:01
Using cached smart_open-7.5.1-py3-none-any.whl (64 kB)
Using cached wrapt-2.1.2-cp311-cp311-macosx_11_0_arm64.whl (61 kB)

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import json, random, hashlib, itertools
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
%matplotlib inline

import networkx as nx
from gensim.models import Word2Vec

In [3]:
# ── Load congress members ─────────────────────────────────────────────────────
with open('congress_network/congress_network_data.json') as f:
    data = json.load(f)[0]
usernames = data['usernameList']
id_to_name = {str(i): usernames[i] for i in range(len(usernames))}
print(f"Congress members: {len(usernames)}")
print(f"Sample: {usernames[:5]}")

# ── Build NetworkX graph ───────────────────────────────────────────────────────
edges_raw = []
with open('congress_network/congress.edgelist') as f:
    for line in f:
        parts = line.strip().split(' ', 2)
        if len(parts) >= 2:
            src, dst = parts[0], parts[1]
            w = float(parts[2].split(':')[1].rstrip('}')) if len(parts) == 3 else 0.0
            edges_raw.append((src, dst, w))

G = nx.DiGraph()
for src, dst, w in edges_raw:
    G.add_edge(src, dst, weight=w)

print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")

Congress members: 475
Sample: ['SenatorBaldwin', 'SenJohnBarrasso', 'SenatorBennet', 'MarshaBlackburn', 'SenBlumenthal']
Nodes: 475
Edges: 13,289


---
# A [63pts].

**Rubric**

[16 pts] Strong/Professional: Correct and complete implementation; reasonable and stated assumptions; thoughtful handling of real-world data (missingness, noise, scale, edge cases); tests are meaningful and the written explanation convincingly argues why they prove correctness; clear, concise prose; code is clean and readable; interpretation goes beyond restating output.

[9 pts] Partial/Developing: Core task mostly complete but with gaps or minor mistakes; tests are present but superficial or poorly justified; reasoning is shallow or mostly descriptive.

[0 pts] Minimal/Incorrect: Task largely incorrect or missing; no meaningful tests; code does not run.

In Homework 9, we used Apache Spark to analyze this same Congressional Twitter network — computing PageRank, in-degree centrality, and word2vec embeddings to understand how information spreads. Those methods produced either a single scalar per node (PageRank, degree) or embeddings over *words* in a text corpus.

Here we go a step further: instead of a scalar, we'll learn a **64-dimensional vector** for every congress member that encodes their position and role in the follow network. These node embeddings power real systems — LinkedIn's 'People You May Know,' Twitter's 'Who to Follow,' fraud detection networks at Stripe. The central question we'll explore through a systematic ablation: **what does it actually mean for two congress members to be 'similar,' and how does that change depending on how we configure the random walk?**

# 1. node2vec p/q Ablation  [16pts]

Using `torch_geometric.nn.Node2Vec`, train embeddings (64-dim, `walk_length=20`, `walks_per_node=10`, `context_size=5`) for all 9 combinations of:

- **p ∈ {0.25, 1, 4}**
- **q ∈ {0.25, 1, 4}**

You will need to build a `edge_index` tensor from the NetworkX graph and write a training loop using `model.loader()` and `torch.optim.SparseAdam`.

Pick **one congress member** with at least 10 followees. For each (p, q) setting, find their top-5 most similar members by cosine similarity. Then:

1. Build a 3×3 table showing the top-5 neighbor lists across settings.
2. Compute pairwise Jaccard similarity between every pair of top-5 lists (i.e., how many members appear in both lists). Which settings produce the most similar neighbor lists? Which are most different?
3. Which parameter — p or q — has a larger effect on the results on this network? Support your answer with evidence from the table.
4. Pick the two most different settings. What does each top-5 list tell you about what "similar" means under that (p, q)?  

**Testing:** Include 2–3 assertions or print-based checks directly in your code cell. In your written answer, explain what tests you wrote and **why those specific tests prove your code is correct**.

*your answer here*

# 2. Visualizing the Ablation  [16pts]

Using the 9 embedding matrices from Q1, run **t-SNE** (2D) on each. Color each point by community (use `nx.algorithms.community.greedy_modularity_communities` on the undirected graph, keeping the 3 largest communities).

1. Arrange the 9 t-SNE plots in a **3×3 grid** with q increasing left→right and p increasing top→bottom. Label each subplot with its (p, q) values.
2. Describe the visual trend across columns (varying q) and across rows (varying p). Does cluster tightness, separation, or shape change? In which direction is the effect larger?
3. Pick the two most visually distinct plots. Identify 2–3 specific congress members who moved significantly between the two projections. What does that movement mean in terms of their role in the network?
4. Based on the grid, which single (p, q) setting would you deploy for a 'People You May Know' feature, and which for a 'Find Similar Accounts' feature? Justify each choice.

**Testing:** Include 2–3 assertions or print-based checks directly in your code cell. In your written answer, explain what tests you wrote and **why those specific tests prove your code is correct**.

*your answer here*

# 3. Biased Walks: Implementing node2vec Sampling from Scratch  [16pts]

The node2vec library handles biased walk sampling internally. Implementing it yourself reveals *why* p and q produce the effects you observed in Q1 and Q2.

**Part A — Implement biased walk sampling:**
Write `node2vec_walk(G, start, p, q, walk_length)` that generates a single biased walk. At each step, classify each neighbor of the current node relative to the previous node:
- Same as previous node → weight `1/p`
- Distance 1 from previous node (shared neighbor) → weight `1`
- Distance 2 from previous node → weight `1/q`

Normalize and sample. For the first step (no previous node), sample uniformly.

**Part B — Characterize walks on the congress network:**
Generate 500 walks of length 20 from the same starting node under `(p=0.25, q=4)`, `(p=1, q=1)`, and `(p=4, q=0.25)`. For each setting compute:
- Fraction of steps that *return* to the previous node
- Fraction of steps that go to a node at distance ≥ 2 from the previous node
- Distribution of unique nodes visited per walk (plot as a histogram)

**Part C — Connect back to Q1/Q2:**
Do the walk statistics explain the patterns you saw in Q1 and Q2? Which statistic (return rate vs. exploration rate) better predicts how different two embedding spaces will be?

**Testing:** Include 2–3 assertions or print-based checks directly in your code cell. In your written answer, explain what tests you wrote and **why those specific tests prove your code is correct**.

*your answer here*

Node embeddings gave us a rich representation of each congress member as a vector. But what if we want to find similar members without embeddings at all — just based on raw behavioral overlap, like who they follow?

With 475 members there are over 112,000 possible pairs. That's manageable now, but real social networks have millions of users, making brute-force comparison infeasible. **Locality-Sensitive Hashing (LSH)** is the industry-standard solution: a probabilistic pipeline that finds similar items in massive datasets without comparing all pairs. It underlies near-duplicate detection at Google, Spotify's music recommendation, and plagiarism detection at scale. Here we'll apply it to the congress follow network and see exactly how the threshold parameter controls the tradeoff between what you find and how much work you do.

# 4. LSH at Different Thresholds  [15pts]

Using `datasketch.MinHash` and `datasketch.MinHashLSH`, apply the LSH pipeline to the Congress Twitter Network. Treat each member's **set of followees** (outgoing neighbors) as their document. Use members with ≥ 10 followees and 128 permutations.

Run `MinHashLSH` at thresholds: **0.1, 0.2, 0.3, 0.5, 0.7**. For each threshold report:
- Number of candidate pairs returned
- Of a random sample of 50 candidate pairs, the fraction with true Jaccard ≥ threshold (precision proxy)
- Of the top-20 truly similar pairs by exact Jaccard, how many were recovered (recall proxy)

Present your results as a table, then:
1. Plot candidate pair count (y-axis, log scale) vs. threshold (x-axis). Describe the shape of the curve.
2. You need to recover all pairs with true Jaccard ≥ 0.4 without missing any. Which threshold do you choose, and what do you give up?
3. At threshold=0.1, is LSH still useful compared to brute force? Why or why not?

**Testing:** Include 2–3 assertions or print-based checks directly in your code cell. In your written answer, explain what tests you wrote and **why those specific tests prove your code is correct**.

*your answer here*

# Resources for Section A

```
On my honor, I declare the following resources:
1. Collaborators:
- ...

2. Web Sources:
- ...

3. AI Tools:
- ...
```

---
# B [30pts]. Interview Questions

We now pretend this is a real job interview. Here's some guidance on how to answer these questions:

1. Briefly restate the question and state any assumptions you are making.
2. Explain your reasoning out loud, focusing on tradeoffs, limitations, and constraints.
3. Keep answers as short and clear as they can be (while still answering the question).
4. Write/speak in a conversational but professional tone.

There may not be a single correct answer. We are grading whether your reasoning is reasonable and aware of limitations.

**Rubric**

[10pt] Clear understanding of the question; reasonable assumptions; thoughtful reasoning that acknowledges tradeoffs and limitations; clear, concise communication.

[5pt] Basic understanding but shallow reasoning or unclear assumptions.

[0pt] Minimal, unclear, or incorrect response.

# 1.
We need graph embeddings for a 'People You May Know' feature — should we use DeepWalk, LINE, or node2vec? What's actually different about them?

*your answer here*

# 2.
We're building a near-duplicate detection system for 50 million documents. Why would you use LSH over just comparing all pairs — especially if LSH can miss things?

*your answer here*

# 3.
(Video) Explain node2vec to me. Specifically — why does the p/q biasing actually matter? Isn't a random walk just a random walk?

*your answer here (include video link)*

# Resources for Section B

```
On my honor, I declare the following resources:
1. Collaborators:
- ...

2. Web Sources:
- ...

3. AI Tools:
- ...
```

---
# C [7pts]. What new questions do you have?
We want you to think bigger! Tell us what questions and curiosity this homework brings up for you.

**Rubric**

[7pt] Complete, thoughtful response.

[4pt] Partial response.

[0pt] Minimal response.

# 1.
What new questions do you have after this homework? Or, what topics are you curious about now? List at least 3.

*your answer here*